In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
import joblib
import os
import optuna
from optuna.samplers import TPESampler

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the data
# Identify categorical and numerical columns
categorical_cols = ['current_stop_name', 'next_stop_name', 'day_of_week', 'weather_condition']
numerical_cols = ['is_holiday', 'is_peak_hour', 'passenger_count', 'current_speed', 
                  'distance_to_next_stop', 'current_lat', 'current_lon']

# Create preprocessors
numerical_scaler = StandardScaler()
# Updated parameter: sparse -> sparse_output
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
X_train_num = numerical_scaler.fit_transform(X_train[numerical_cols])
X_test_num = numerical_scaler.transform(X_test[numerical_cols])

X_train_cat = categorical_encoder.fit_transform(X_train[categorical_cols])
X_test_cat = categorical_encoder.transform(X_test[categorical_cols])

# Combine numerical and categorical features
X_train_processed = np.hstack((X_train_num, X_train_cat))
X_test_processed = np.hstack((X_test_num, X_test_cat))

# Reshape data for CNN (add channel dimension)
X_train_cnn = X_train_processed.reshape(X_train_processed.shape[0], X_train_processed.shape[1], 1)
X_test_cnn = X_test_processed.reshape(X_test_processed.shape[0], X_test_processed.shape[1], 1)

# Function to evaluate model
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")
    
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Define baseline CNN model
def create_baseline_cnn_model(input_shape):
    model = keras.Sequential([
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(50, activation='relu'),
        layers.Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train baseline CNN model
print("Training baseline CNN model...")
baseline_model = create_baseline_cnn_model((X_train_cnn.shape[1], 1))

# Use early stopping to prevent overfitting
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

baseline_history = baseline_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate baseline model
print("\nBaseline Model Performance:")
y_pred_baseline = baseline_model.predict(X_test_cnn).flatten()
baseline_metrics = evaluate_model(y_test, y_pred_baseline)

# Optuna optimization for CNN
def objective(trial):
    # Define hyperparameters to optimize
    filters1 = trial.suggest_int('filters1', 16, 128)
    filters2 = trial.suggest_int('filters2', 16, 128)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dense_units = trial.suggest_int('dense_units', 16, 128)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    
    # Build model with the suggested hyperparameters
    model = keras.Sequential([
        layers.Conv1D(filters=filters1, kernel_size=kernel_size, activation='relu', 
                     input_shape=(X_train_cnn.shape[1], 1)),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Conv1D(filters=filters2, kernel_size=kernel_size, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1)
    ])
    
    # Compile model
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Define callbacks
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    # Train model
    history = model.fit(
        X_train_cnn, y_train,
        epochs=100,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Get the validation loss from the epoch with the best performance
    val_loss = min(history.history['val_loss'])
    
    return val_loss

print("\nStarting Optuna optimization...")
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)

# Get the best hyperparameters
best_params = study.best_params
print("\nBest parameters:", best_params)

# Train optimized model with the best hyperparameters
print("\nTraining optimized model with best parameters...")
optimized_model = keras.Sequential([
    layers.Conv1D(filters=best_params['filters1'], kernel_size=best_params['kernel_size'], 
                 activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Conv1D(filters=best_params['filters2'], kernel_size=best_params['kernel_size'], 
                 activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Flatten(),
    layers.Dense(best_params['dense_units'], activation='relu'),
    layers.Dropout(best_params['dropout_rate']),
    layers.Dense(1)
])

# Compile model
optimizer = optimizers.Adam(learning_rate=best_params['learning_rate'])
optimized_model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Train model
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

optimized_history = optimized_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate optimized model
print("\nOptimized Model Performance:")
y_pred_optimized = optimized_model.predict(X_test_cnn).flatten()
optimized_metrics = evaluate_model(y_test, y_pred_optimized)

# Compare baseline and optimized models
print("\nPerformance Comparison:")
print(f"Baseline RMSE: {baseline_metrics['RMSE']:.4f}")
print(f"Optimized RMSE: {optimized_metrics['RMSE']:.4f}")
print(f"Improvement: {(baseline_metrics['RMSE'] - optimized_metrics['RMSE']):.4f} ({((baseline_metrics['RMSE'] - optimized_metrics['RMSE'])/baseline_metrics['RMSE']*100):.2f}%)")

# Save the optimized model - Added file extension
model_filename = 'optimized_cnn_eta_predictor.keras'  # Use .keras extension for TF 2.x
optimized_model.save(model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Save an alternative format if needed
h5_model_filename = 'optimized_cnn_eta_predictor.h5'  # H5 format is also supported
optimized_model.save(h5_model_filename)
print(f"Optimized model also saved as {h5_model_filename}")

# Save the preprocessors for later use
preprocessor_filename = 'eta_preprocessors.pkl'
joblib.dump({
    'numerical_scaler': numerical_scaler,
    'categorical_encoder': categorical_encoder,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols
}, preprocessor_filename)
print(f"Preprocessors saved as {preprocessor_filename}")

# Feature importance is not directly available for neural networks like in tree-based models
# Alternative approach: Permutation importance
print("\nNote: Feature importance for CNN models requires additional techniques like permutation importance analysis")

# Add code to implement permutation importance (optional)
# This gives you feature importance similar to tree-based models
try:
    from sklearn.inspection import permutation_importance
    
    print("\nCalculating permutation feature importance...")
    # First, we need a prediction function that works with the original features
    def predict_func(X_subset):
        # Process the input data exactly as we did during training
        X_subset_num = numerical_scaler.transform(X_subset[numerical_cols])
        X_subset_cat = categorical_encoder.transform(X_subset[categorical_cols])
        X_subset_processed = np.hstack((X_subset_num, X_subset_cat))
        X_subset_cnn = X_subset_processed.reshape(X_subset_processed.shape[0], X_subset_processed.shape[1], 1)
        return optimized_model.predict(X_subset_cnn).flatten()
    
    # Calculate permutation importance
    r = permutation_importance(predict_func, X_test, y_test, 
                              n_repeats=10, random_state=42, n_jobs=-1)
    
    # Create DataFrame of feature importances
    feature_importance = pd.DataFrame({
        'feature': features,
        'importance': r.importances_mean
    }).sort_values('importance', ascending=False)
    
    print("\nPermutation Feature Importance:")
    print(feature_importance)
    
    # Save feature importance to CSV
    feature_importance.to_csv('cnn_feature_importance.csv', index=False)
    print("Feature importance saved to cnn_feature_importance.csv")
    
except Exception as e:
    print(f"\nCould not calculate permutation importance: {e}")
    print("To implement feature importance for the CNN model, consider using permutation_importance from sklearn.inspection")

Training baseline CNN model...
Epoch 1/100


C:\Users\cheng\AppData\Local\Temp\ipykernel_17120\379497229.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_17120\379497229.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_peak_hour'] = X['is_peak_hour'].astype(int)
c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a 

2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.9316 - mae: 0.7660 - val_loss: 0.4873 - val_mae: 0.4286
Epoch 2/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.4035 - mae: 0.4216 - val_loss: 0.3252 - val_mae: 0.3818
Epoch 3/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3790 - mae: 0.4052 - val_loss: 0.3587 - val_mae: 0.3950
Epoch 4/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3649 - mae: 0.3987 - val_loss: 0.3039 - val_mae: 0.3660
Epoch 5/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3610 - mae: 0.3941 - val_loss: 0.3664 - val_mae: 0.4136
Epoch 6/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3571 - mae: 0.3923 - val_loss: 0.4383 - val_mae: 0.3985
Epoch 7/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3536 - mae: 0.3863 - val_loss: 0.3458 - val_mae: 0.3854
Epoch 8/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3491 - mae: 0.3790 - val_loss: 0.2946 - val_mae: 0.3568
Epoch 9/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 3

[I 2025-04-11 15:39:37,763] A new study created in memory with name: no-name-63aca875-beed-4cdf-a2be-7cfcb45b6603


MAE: 0.3466
MSE: 0.3001
RMSE: 0.5478
R2 Score: 0.9639

Starting Optuna optimization...


c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2025-04-11 15:41:02,695] Trial 0 finished with value: 0.31691187620162964 and parameters: {'filters1': 58, 'filters2': 123, 'kernel_size': 4, 'dense_units': 83, 'learning_rate': 0.0002051338263087451, 'batch_size': 64, 'dropout_rate': 0.3832290311184182}. Best is trial 0 with value: 0.31691187620162964.
[I 2025-04-11 15:41:38,202] Trial 1 finished with value: 0.4336099624633789 and parameters: {'filters1': 18, 'filters2': 125, 'kernel_size': 5, 'dense_units': 39, 'learning_rate': 0.0002310201887845295, 'batch_size': 64, 'dropout_rate': 0.21649165607921678}. Best is trial 0 with value: 0.31691187620162964.
[I 202


Best parameters: {'filters1': 75, 'filters2': 58, 'kernel_size': 4, 'dense_units': 60, 'learning_rate': 0.000632751642127369, 'batch_size': 128, 'dropout_rate': 0.11737661492901214}

Training optimized model with best parameters...
Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 4.7262 - mae: 1.2773 - val_loss: 0.5614 - val_mae: 0.4665
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.9002 - mae: 0.5824 - val_loss: 0.4208 - val_mae: 0.4129
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7377 - mae: 0.5312 - val_loss: 0.4494 - val_mae: 0.4338
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6654 - mae: 0.5135 - val_loss: 0.3543 - val_mae: 0.3900
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6288 - mae: 0.4929 - val_loss: 0.3517 - val_mae: 0.3998
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5867 - mae: 0.4789 - val_loss: 0.3641 - val_mae: 0.3942
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step 


Optimized model saved as optimized_cnn_eta_predictor.keras
Optimized model also saved as optimized_cnn_eta_predictor.h5
Preprocessors saved as eta_preprocessors.pkl

Note: Feature importance for CNN models requires additional techniques like permutation importance analysis

Calculating permutation feature importance...

Could not calculate permutation importance: The 'estimator' parameter of permutation_importance must be an object implementing 'fit'. Got <function predict_func at 0x000001E2F19D08B0> instead.
To implement feature importance for the CNN model, consider using permutation_importance from sklearn.inspection
